# Jour 3 · Détecter des anomalies avec des règles simples


## Objectifs

- partir d'un seuil métier avant un algorithme
- calculer un Z-score mobile sans regarder le futur
- comparer précision et rappel

Une anomalie peut être une valeur impossible, un écart au comportement récent ou une combinaison inhabituelle. Il n'existe pas de méthode universelle : nous commençons par des règles compréhensibles.

![Technicien contrôlant la vibration d'un moteur HVAC à partir d'anomalies visibles sur plusieurs capteurs](../assets/jour_03/00_detection_maintenance_hvac.png)

*La détection n'est utile que si elle relie un comportement inhabituel des données à une investigation physique contextualisée.*

![Comportement normal, pic ponctuel, anomalie contextuelle et dérive progressive](../assets/jour_03/01_formes_anomalies.png)

*Une anomalie n'est pas toujours un pic : sa forme détermine souvent la méthode la plus adaptée.*

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.metrics import precision_score, recall_score, f1_score


from pathlib import Path


def find_project_root() -> Path:
    for candidate in (Path.cwd(), *Path.cwd().parents):
        if (candidate / "datasets").is_dir():
            return candidate
    raise FileNotFoundError("Dossier datasets introuvable. Lancez Jupyter depuis le projet.")


ROOT = find_project_root()
DATA_DIR = ROOT / "datasets"


plt.style.use("seaborn-v0_8-whitegrid")
df = pd.read_csv(DATA_DIR / "prepared" / "iot_hvac_labeled.csv")
df["timestamp"] = pd.to_datetime(df["timestamp"], utc=True)
df = df.set_index("timestamp").sort_index()

print(df["is_anomaly"].value_counts())
print("\nTypes injectés :")
print(df.loc[df["is_anomaly"] == 1, "anomaly_type"].value_counts())

In [ ]:
def metrics(truth, flags):
    return {
        "alerts": int(flags.sum()),
        "precision": precision_score(truth, flags, zero_division=0),
        "recall": recall_score(truth, flags, zero_division=0),
        "f1": f1_score(truth, flags, zero_division=0),
    }


domain_flag = (
    (df["temperature_c"] < 15)
    | (df["temperature_c"] > 28)
    | (df["vibration_mm_s"] > 2.2)
    | (df["pressure_bar"] < 2.25)
).astype(int)
pd.Series(metrics(df["is_anomaly"], domain_flag), name="seuils métier")

Pour un Z-score mobile, la moyenne et l'écart-type de référence sont décalés d'un pas avec `shift(1)`. La mesure courante ne participe donc pas à sa propre référence et aucune mesure future n'est utilisée.

![Comparaison entre un seuil métier fixe et une référence mobile fondée sur le passé](../assets/jour_03/01_seuil_metier_et_zscore.png)

*Le seuil métier encode une limite connue ; le Z-score mobile demande si la mesure s'écarte fortement de son contexte récent.*

In [ ]:
signal = df["vibration_mm_s"]
past_mean = signal.shift(1).rolling(96 * 2, min_periods=96).mean()
past_std = signal.shift(1).rolling(96 * 2, min_periods=96).std()
df["vibration_zscore"] = ((signal - past_mean) / past_std).abs()
zscore_flag = (df["vibration_zscore"] > 4).fillna(False).astype(int)

comparison = pd.DataFrame({
    "seuils métier": metrics(df["is_anomaly"], domain_flag),
    "Z-score mobile": metrics(df["is_anomaly"], zscore_flag),
}).T
comparison

In [ ]:
view = df.last("14D")
flags_view = zscore_flag.loc[view.index] == 1
ax = view["vibration_mm_s"].plot(figsize=(13, 4), label="vibration")
ax.scatter(view.index[flags_view], view.loc[flags_view, "vibration_mm_s"], color="red", label="alerte Z-score")
ax.set_ylabel("mm/s")
ax.set_title("Détection d'écarts au comportement récent")
ax.legend()
plt.show()

### À vous de jouer — choisir le seuil du Z-score

Comparez les seuils 3, 4 et 5 dans un tableau. Observez le compromis entre nombre d'alertes, précision et rappel.

In [ ]:
# Écrivez votre code ici.
pass

### À vous de jouer — combiner des règles

Créez une alerte lorsque les seuils métier OU le Z-score signalent un problème. Mesurez cette combinaison.

**Indice :** L'opérateur | représente le OU entre deux séries booléennes.

In [ ]:
# Écrivez votre code ici.
pass

## À retenir

Une règle simple, explicable et surveillée est souvent un excellent premier détecteur. Son seuil doit être relié au coût des faux positifs et des anomalies manquées.